In [ ]:
import sagemaker
from sagemaker.model import Model
import boto3

# ----------------------------
# 1️⃣ Setup session & role
# ----------------------------
sess = sagemaker.Session()
role = role = sagemaker.get_execution_role()  # replace with your SageMaker execution role
endpoint_name = "clip-endpoint-byoc"
# ----------------------------
# 2️⃣ ECR image URI
# ----------------------------
image_uri = "your_ECR_imageUrl"

# ----------------------------
# 3️⃣ Create SageMaker Model
# ----------------------------
clip_model = Model(
    image_uri=image_uri,
    role=role,
    sagemaker_session=sess
)

# ----------------------------
# 4️⃣ Deploy Endpoint
# ----------------------------
predictor = clip_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",  # CPU instance. For GPU: "ml.g4dn.xlarge"
    endpoint_name=endpoint_name
)

print(f"✅ Endpoint deployed")

autoscaling_client = boto3.client("application-autoscaling")

resource_id = f"endpoint/{endpoint_name}/variant/AllTraffic"

# Register endpoint for scaling
autoscaling_client.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=1,
    MaxCapacity=3
)

# Attach scaling policy (based on invocations per instance)
autoscaling_client.put_scaling_policy(
    PolicyName="InvocationsAutoScalingPolicy",
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 70.0,  # target 70 invocations per instance
        "PredefinedMetricSpecification": {
            "PredefinedMetricType": "SageMakerVariantInvocationsPerInstance"
        },
        "ScaleInCooldown": 120,
        "ScaleOutCooldown": 60
    }
)